In [ ]:
import os
import sys
import time
import json
import random
import argparse
from pathlib import Path

import torch

# Repo setup
repo_root = "/scratch/jq2uw/MME/reasonedit"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from argparse import Namespace
from pathlib import Path
import torch

from revlm.config_utils import configure_args
from revlm.run.edit import run_edit

editor_name = "reasonedit"
model_name = "qwen3_4b"       # <-- change to match your model
dataset_name = "aokvqa"        # <-- change to match your dataset


In [ ]:
import shutil

# Map short model names to full HF model names used in results/pred/
MODEL_NAME_MAP = {
    "qwen3": "Qwen3-VL-8B-Instruct",
    "qwen3_4b": "Qwen3-VL-4B-Instruct",
    "llava": "llava-1.5-7b-hf",
    "blip": "instructblip-vicuna-7b",
}

project_root = Path(repo_root)

full_model_name = MODEL_NAME_MAP[model_name]
src = project_root / "results" / "pred" / full_model_name / dataset_name / "mc_all.json"
dst_dir = project_root / "results" / "test" / f"editor_{editor_name}" / model_name / dataset_name
dst_dir.mkdir(parents=True, exist_ok=True)
dst = dst_dir / "pred_mc.json"

assert src.exists(), f"Source not found: {src}"
shutil.copy2(src, dst)
print(f"Copied {src}\n    -> {dst}")


In [ ]:

# Project root
project_root = Path(repo_root)

# Build test paths from args (no hardcoding)
args = Namespace(
    config="revlm/config/config.yaml",
    editor=editor_name,
    model_name=model_name,
    dataset_name=dataset_name,
    task="mc",
    batch_size=1,
    split="all",
    rationale=False,
    cot=False,
    subsample=0,
    subsample_edits=500,
    overwrite=True,
)

# Derive result prefix from args
res_prefix = project_root / "results" / "test" / f"editor_{args.editor}" / args.model_name / args.dataset_name
res_prefix.mkdir(parents=True, exist_ok=True)
(res_prefix / "pred_postedit").mkdir(parents=True, exist_ok=True)

# Force all outputs into the test prefix (overwrite allowed)
args.task_dir = str(res_prefix)
args.edit_dir = str(res_prefix)
args.pred_dir = str(res_prefix)
args.pred_path = str(res_prefix / "pred_mc.json")

args.pred_postedit_dir = str(res_prefix / f"pred_postedit")

args.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
args.suffix = "_cot" if args.rationale and args.cot else ("_rationale" if args.rationale else "")

config = configure_args(args, config_path=args.config)
config.subsample = args.subsample
config.subsample_edits = args.subsample_edits
config.rationale = args.rationale
config.cot = args.cot
config.pred_path = args.pred_path
config.overwrite = args.overwrite
config.task_dir = args.task_dir
config.pred_dir = args.pred_dir
config.pred_postedit_dir = args.pred_postedit_dir
config.edit_dir = args.edit_dir

config.plot_k_dist = True 
run_edit(config, sequential=True, eval_every=10)